# Free pattern-quality pilot
Use an eligible free Colab GPU with the same account. If no free GPU is available, stop and retry later. No paid services, keys, keep-alives or quota workarounds are used.
This pilot uses 27 existing approved synthetic targets that pass the current compiler and conservative clearance checks, with three whole families held out for development evaluation. Two scenes with overlapping nominal bead envelopes are excluded. Targets are not physically tested.
First evaluate the base model with improved spatial instructions. Then train a three-epoch LoRA and compare it on the same twelve annotated development prompts. All predictions and checkpoints are saved to Drive. Geometry quality, feature coverage and physical buildability remain distinct.


In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, torch
if not torch.cuda.is_available():
    raise RuntimeError("No eligible free CUDA GPU. Stop and retry later; do not upgrade.")
from google.colab import drive
drive.mount("/content/drive")
repo=Path("/content/beadform-quality-pilot")
if not repo.exists():
    subprocess.run(["git","clone","--depth","1","--branch","runpod-training-guide","https://github.com/c-tustin/beadform.git",str(repo)],check=True)
run=Path("/content/drive/MyDrive/Beadform-free-qwen/runs/quality-pilot-v1")
run.mkdir(parents=True,exist_ok=True)
system=repo/"training/system-quality.txt"
data=repo/"training/quality-pilot-v1"
benchmark=repo/"training/benchmarks/schema-fresh-12.jsonl"
requirements=repo/"training/benchmarks/pattern-quality-12.jsonl"
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
metadata={"repo_commit":subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip(),"model":"Qwen/Qwen3-VL-2B-Instruct","epochs":3,"parts_first":True,"system_sha256":sha(system),"train_sha256":sha(data/"train.jsonl"),"eval_sha256":sha(data/"eval.jsonl"),"benchmark_sha256":sha(benchmark),"requirements_sha256":sha(requirements),"gpu":torch.cuda.get_device_name(0),"note":"Text-only pilot; no physical success labels. These are development benchmarks."}
manifest=run/"run-metadata.json"
if manifest.exists() and json.loads(manifest.read_text())!=metadata:
    raise RuntimeError("Saved configuration differs. Restore its code revision or choose a new run directory.")
manifest.write_text(json.dumps(metadata,indent=2)+"\n")
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(repo/"training/requirements.txt")],check=True)
freeze=subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True)
if not (run/"pip-freeze.txt").exists():
    (run/"pip-freeze.txt").write_text(freeze)
print(json.dumps(metadata,indent=2))


In [ ]:
def logged(command,label):
    print("START",label,flush=True)
    with (run/(label+".log")).open("a") as log:
        process=subprocess.Popen(command,cwd=repo,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in process.stdout:
            log.write(line); log.flush(); print(line,end="",flush=True)
        code=process.wait()
    if code:
        raise RuntimeError(f"{label} exited {code}; inspect its saved log.")
def evaluate(label,adapter=None):
    destination=run/label
    destination.mkdir(exist_ok=True)
    summary=destination/"summary.json"
    if not summary.exists():
        command=[sys.executable,"-u",str(repo/"training/evaluate_lora.py"),"--data",str(benchmark),"--system",str(system),"--output",str(destination/"predictions.jsonl"),"--summary",str(summary),"--bounded","--prompts-only","--resume"]
        if adapter is not None:
            command += ["--adapter",str(adapter)]
        logged(command,label)
    quality=destination/"pattern-quality.json"
    if not quality.exists():
        logged(["node",str(repo/"training/evaluate_pattern_quality.cjs"),"--predictions",str(destination/"predictions.jsonl"),"--requirements",str(requirements),"--output",str(quality)],label+"-quality")
    print(label,"SCHEMA",summary.read_text(),flush=True)
    print(label,"PATTERN",json.dumps(json.loads(quality.read_text())["summary"]),flush=True)
logged([sys.executable,str(repo/"training/train_lora.py"),"--train",str(data/"train.jsonl"),"--eval",str(data/"eval.jsonl"),"--system",str(system),"--check"],"data-check")
evaluate("base-quality-prompt")


In [ ]:
adapter=run/"adapter"
if not (run/"training-complete.json").exists():
    logged([sys.executable,"-u",str(repo/"training/train_lora.py"),"--train",str(data/"train.jsonl"),"--eval",str(data/"eval.jsonl"),"--system",str(system),"--parts-first","--epochs","3","--save-steps","1","--model","Qwen/Qwen3-VL-2B-Instruct","--output",str(adapter),"--resume"],"training")
    (run/"training-complete.json").write_text(json.dumps({"adapter":str(adapter),"physically_tested":False})+"\n")
evaluate("adapter-quality-prompt",adapter)
print("PILOT COMPLETE",run,flush=True)
print("Compare requested-feature coverage and clearance as well as schema validity. A higher schema score alone is insufficient.")
